# tensor-item-scalar — ex6: training loop with .item() logging and loss curve

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-item-scalar`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-item-scalar`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-item-scalar"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch `.item()` — quick refresher

`tensor.item()` extracts a Python scalar (`int`, `float`, or `bool`) from a 0-D or 1-element tensor. It detaches from the autograd graph and pulls the value back to CPU. This is the canonical bridge from tensor-world to Python-world: logging, control flow, plotting, and stop conditions all need scalars.

**Calling `.item()` on a multi-element tensor raises.** Use `.tolist()` if you want every element as a Python list. Calling `.item()` inside a hot inner loop forces a CPU sync — fine for diagnostics, expensive in the training step itself.

### Exercise 6 — training loop with .item() logging and loss curve

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Combine a 3-step gradient-descent loop with `.item()` extraction to build a Python-side `losses` list and plot the loss curve.
> Keywords: training-loop, logging, loss-curve, visualization
> ```

**KCs targeted:** `item-zero-d-extract`, `item-control-flow`

Implement `ex6_train(x_init, target, lr, n_steps)`. A minimal gradient-descent loop that demonstrates the canonical use of `.item()` for logging:

1. Start from `x = x_init.clone().detach().requires_grad_(True)`.
2. For `n_steps` iterations:
   a. Compute `loss = ((x - target) ** 2).sum()`. (Mean-squared error against the target.)
   b. **Append `loss.item()` to a Python list** — this is the scalar extraction. Without `.item()` you'd accumulate a graph of tensors and leak memory.
   c. Manually zero `x.grad` if it exists, call `loss.backward()`, and update `x.data -= lr * x.grad`.
3. Return `(x.detach(), losses_list)`.

Inputs:
- `x_init`: starting 1-D float tensor.
- `target`: same-shape target tensor.
- `lr`: float learning rate.
- `n_steps`: int.

Output: tuple `(x_final, losses)`. `x_final` is a detached tensor; `losses` is a list of `n_steps` Python floats.

The visualization plots the loss curve so you can see the convergence (or divergence) of your loop.

In [ ]:
def ex6_train(x_init: Tensor, target: Tensor, lr: float, n_steps: int) -> tuple:
    """3-step GD with .item() logging. Returns (x_final, losses)."""
    raise NotImplementedError()


def _test_ex6():
    x_init = t.tensor([5.0, -3.0, 2.0])
    target = t.tensor([0.0, 0.0, 0.0])
    x_final, losses = ex6_train(x_init, target, lr=0.1, n_steps=5)
    # Type checks.
    assert isinstance(losses, list), f'losses must be a Python list, got {type(losses)}'
    assert len(losses) == 5, f'expected 5 losses, got {len(losses)}'
    for i, lv in enumerate(losses):
        assert isinstance(lv, float), f'losses[{i}] must be a Python float, got {type(lv)}'
    assert isinstance(x_final, Tensor)
    assert not x_final.requires_grad, 'x_final should be detached'
    # Loss must decrease monotonically for this convex problem.
    for i in range(1, 5):
        assert losses[i] < losses[i-1], (
            f'loss not decreasing at step {i}: {losses[i-1]:.4f} → {losses[i]:.4f}'
        )
    # First loss should equal sum((x_init - target)**2) = 25 + 9 + 4 = 38.
    assert abs(losses[0] - 38.0) < 1e-4, f'expected first loss 38.0, got {losses[0]}'
    # After 5 steps with lr=0.1, x should be much closer to target.
    assert (x_final.abs().max().item() < x_init.abs().max().item()), 'x should approach target'

    # --- Longer-horizon loss curve visualization ---
    x_big = t.tensor([10.0, -7.0, 3.0, 5.0, -2.0])
    target_big = t.zeros(5)
    _, big_losses = ex6_train(x_big, target_big, lr=0.05, n_steps=40)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(big_losses, marker='o', markersize=3, color='darkblue')
    ax.set_xlabel('step')
    ax.set_ylabel('loss (sum of squares)')
    ax.set_title(f'ex6 training-loop loss curve (lr=0.05, n_steps=40)')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_train(x_init: Tensor, target: Tensor, lr: float, n_steps: int) -> tuple:
    x = x_init.clone().detach().requires_grad_(True)
    losses = []
    for _ in range(n_steps):
        loss = ((x - target) ** 2).sum()
        losses.append(loss.item())  # scalar extract — no graph leak
        if x.grad is not None:
            x.grad.zero_()
        loss.backward()
        with t.no_grad():
            x -= lr * x.grad
    return x.detach(), losses
```

**Why `.item()` and not just `losses.append(loss)`?** A bare `loss` is a 0-D tensor still wired into the autograd graph. Append 1000 of them and you're holding 1000 graphs in memory, which is exactly the leak that catches every PyTorch beginner. `.item()` cuts the graph and returns a plain Python float.

**Why detach `x` at the end.** Returning a tensor with `requires_grad=True` invites the caller to accidentally chain more autograd onto an old graph. `.detach()` returns a fresh view with no graph history.

**Why `with t.no_grad()` around the update.** The update is a Python operation that mutates `x.data`. Without `no_grad`, it'd be tracked as an op in the next backward — wrong. Inside the block, autograd ignores the write.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()